# Track A v3 — modulation radar, domaine temporel UNIQUEMENT

**Règle Track A :** aucune FFT / STFT / ondelette / spectrogramme codé à la main. Utilisé : normalisation RMS, I/Q, produits conjugués à retard, **produits conjugués d'ordre 2** (compression de chirp), convolutions 1D **apprises** (init aléatoire).

**Nouveautés v3 (par rapport à la v2, v1 = 0.8656 en validation) :**
1. **Branche « banque de filtres appris »** (`USE_FB`) : longs filtres FIR appris sur I/Q (k=256) + détecteur d'énergie |.|. Donne un gain d'intégration cohérent, c'est ce qui manque à bas SNR.
2. **Produits d'ordre 2** (`USE_HAF`) : `p_k[n]·conj(p_k[n-m])` transforme un chirp (FMCW/NLFM/HYP) en phase quasi constante.
3. **Ensemble de 3 modèles**, diagnostic par SNR et matrices de confusion (nouvelle cellule).
4. Dossier de sortie séparé `final_v3` (la v2 n'est pas écrasée).

⚠️ La branche `USE_FB` est une banque de filtres temporelle apprise : légale d'après « convolutions 1D apprises », mais proche d'un spectrogramme. **À confirmer avec les organisateurs** ; mettre `USE_FB = False` sinon.


## 1. Configuration

In [1]:
import csv, json, math, time, copy, hashlib, importlib.util
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F

# ---------- chemins : on remonte jusqu'au dossier qui contient dataset/ et dataset_infos/ ----------
def find_root(start=Path.cwd().resolve()):
    for p in [start, *start.parents]:
        if (p / "dataset").is_dir() and (p / "dataset_infos").is_dir():
            return p
    raise FileNotFoundError("Lance le notebook depuis Radar_ML_lite (dossier contenant dataset/ et dataset_infos/)")

ROOT      = find_root()
DATA_DIR  = ROOT / "dataset"
SPLIT_CSV = ROOT / "dataset_infos" / "split_manifest.csv"
OUT_DIR   = ROOT / "track_a_runs" / "final_v3"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------- réglages ----------
EPOCHS      = 80        # 1 époque = 1 passage sur les 52 800 impulsions (bruit différent à chaque fois)
N_MODELS    = 3         # taille de l'ensemble (chaque modèle ~ 55 min pour 80 époques sur une RTX 4090)
SEEDS       = [42, 43, 44, 45][:N_MODELS]
CURR_EPOCHS = 12        # durée du curriculum de SNR
LOW_FOCUS   = 0.5       # après le curriculum : part des tirages dans [-30,-10] dB
BATCH       = 256
LR          = 2e-3
WEIGHT_DECAY= 0.05
EMA_DECAY   = 0.999
LABEL_SMOOTH= 0.05
LOGIT_ADJ   = 1.0       # tau du logit adjustment (1.0 = équilibre les classes)
LAGS        = (1, 2, 4, 8, 16, 32)  # retards des produits conjugués
USE_HAF     = True                  # produits conjugués d'ordre 2 (compression de chirp)
HAF_PAIRS   = ((4, 4), (8, 8), (16, 16))   # (k, m)
USE_FB      = True                  # branche banque de filtres appris (voir intro : à confirmer avec les organisateurs)
FB_K, FB_STRIDE, FB_F = 256, 16, 64 # noyau, pas, nb de filtres complexes
EVAL_EVERY  = 4         # évaluation rapide (sous-ensemble val) toutes les N époques
VAL_SUBSET  = 1500      # nb d'impulsions val utilisées pendant l'entraînement
SEED        = 42

# ---------- constantes du challenge ----------
FS = 120e6
CLASSES = ["barker", "cw", "fmcw", "fsk", "hyp", "nlfm", "quad"]
SNRS = [30, 25, 20, 15, 10, 5, 0, -5, -10, -15, -20, -25, -30]
N0 = int(20e-6 * FS)                      # fin du bruit seul (20 µs)
S0, S1 = int(21e-6 * FS), int(119e-6 * FS)  # fenêtre signal (21-119 µs) : on ne garde QUE ça

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED); np.random.seed(SEED)
torch.backends.cudnn.benchmark = True
print("device:", device, "| ROOT:", ROOT)

device: cuda | ROOT: C:\Users\Think pro\Desktop\Radar_ML_Lite_DATATHON\Radar_ML_Lite_supervised_multiclass_classification\Radar_ML_lite


## 2. Chargement des données (split officiel)
On lit les pulses train/validation du manifest. Le label vient du nom de fichier. On stocke en int16 uniquement la fenêtre 21–119 µs, plus la puissance de bruit/signal de chaque impulsion (nécessaire pour ajouter du bruit comme les organisateurs).

In [2]:
def get_label(filename):
    token = filename.split("_")[1].split("#")[0].lower()
    return "barker" if token.startswith("barker") else token

def load_split(split_name):
    rows = [r for r in csv.DictReader(open(SPLIT_CSV, newline="")) if r["split"] == split_name]
    by_file = {}
    for r in rows:
        by_file.setdefault(r["file"], []).append(int(r["pulse_index"]))
    X, y, npw, spw = [], [], [], []
    for f, idxs in by_file.items():
        idxs = np.array(sorted(idxs))
        arr = np.load(DATA_DIR / f, mmap_mode="r")
        raw = np.asarray(arr[0][idxs])                      # (n, 33600) int16 entrelacé I,Q,I,Q...
        rf = raw.astype(np.float32)
        i, q = rf[:, 0::2], rf[:, 1::2]
        p = i * i + q * q
        n = p[:, :N0].mean(1)
        s = p[:, S0:S1].mean(1) - n
        ok = np.isfinite(s) & (s > 0)
        X.append(raw[ok][:, 2 * S0:2 * S1].copy())          # fenêtre signal seulement
        y.append(np.full(ok.sum(), CLASSES.index(get_label(f)), np.int64))
        npw.append(n[ok]); spw.append(s[ok])
    return (np.concatenate(X), np.concatenate(y), np.concatenate(npw).astype(np.float32), np.concatenate(spw).astype(np.float32))

t0 = time.time()
Xtr, ytr, ntr, str_ = load_split("train")
Xva, yva, nva, sva  = load_split("validation")
print(f"train {len(Xtr)}  val {len(Xva)}  ({time.time()-t0:.0f}s)")
print("classes train:", dict(zip(CLASSES, np.bincount(ytr, minlength=7).tolist())))

# tout sur le GPU si possible (≈2.5 Go) : plus aucun goulot CPU pendant l'entraînement
dev = torch.device(device)
Xtr_t, ytr_t, ntr_t, str_t = [torch.from_numpy(a).to(dev) for a in (Xtr, ytr, ntr, str_)]
Xva_t, yva_t, nva_t, sva_t = [torch.from_numpy(a).to(dev) for a in (Xva, yva, nva, sva)]
prior = torch.tensor(np.bincount(ytr, minlength=7) / len(ytr), dtype=torch.float32, device=dev)
LOG_PRIOR = torch.log(prior)

train 52800  val 13200  (41s)
classes train: {'barker': 5760, 'cw': 960, 'fmcw': 7680, 'fsk': 15360, 'hyp': 7680, 'nlfm': 7680, 'quad': 7680}


## 3. Bruit et features (100 % domaine temporel)

In [3]:
def add_noise(x, npw, spw, snr_db, gen=None):
    """Équivalent torch de degrade() : ajoute du bruit gaussien pour atteindre le SNR demandé.
    x : (B, 2L) float entrelacé I,Q ; snr_db : float ou tensor (B,)."""
    if not torch.is_tensor(snr_db):
        snr_db = torch.full_like(npw, float(snr_db))
    added = spw / torch.pow(10.0, snr_db / 10.0) - npw
    sigma = torch.sqrt(torch.clamp(added, min=0) / 2).unsqueeze(1)
    noise = torch.randn(x.shape, device=x.device, generator=gen) * sigma
    return torch.clamp(torch.trunc(x + noise), -32768, 32767)

def features_from_z(z, rotate=False):
    """z : (B, L) complexe -> (B, 2 + 2*len(LAGS), L).
    1) division par la RMS de l'impulsion  2) I, Q  3) produits conjugués à retard k (phase relative)."""
    z = z / (z.abs().pow(2).mean(1, keepdim=True).sqrt() + 1e-6)
    if rotate:  # phase de porteuse aléatoire : les produits conjugués n'en dépendent pas, I/Q oui
        th = torch.rand(z.shape[0], 1, device=z.device) * (2 * math.pi)
        z = z * torch.polar(torch.ones_like(th), th)
    ch = [z.real, z.imag]
    for k in LAGS:
        p = z[:, k:] * torch.conj(z[:, :-k])
        p = F.pad(p, (k, 0))
        ch += [p.real, p.imag]
    if USE_HAF:
        for k, m in HAF_PAIRS:
            p = F.pad(z[:, k:] * torch.conj(z[:, :-k]), (k, 0))
            q = F.pad(p[:, m:] * torch.conj(p[:, :-m]), (m, 0))
            ch += [q.real, q.imag]
    return torch.stack(ch, dim=1)

def features(x, rotate=False):
    """x : (B, 2L) entrelacé -> features."""
    return features_from_z(torch.complex(x[:, 0::2], x[:, 1::2]), rotate)

N_CH = 2 + 2 * len(LAGS) + (2 * len(HAF_PAIRS) if USE_HAF else 0)
print("canaux d'entrée:", N_CH, "| longueur:", S1 - S0)

canaux d'entrée: 20 | longueur: 11760


## 4. Modèle
Branche A : stem conv (k=31) → 6 étages résiduels (stride 2) + Squeeze-Excite sur les canaux I/Q + produits conjugués.
Branche B (`USE_FB`) : banque de 64 filtres complexes appris (k=256) sur I/Q → module → 3 étages résiduels → même longueur temporelle que A.
Fusion par concaténation → GRU bidirectionnel → attentive statistics pooling → MLP.

In [4]:
class SE(nn.Module):
    def __init__(self, c, r=8):
        super().__init__()
        self.fc1, self.fc2 = nn.Linear(c, max(c // r, 8)), nn.Linear(max(c // r, 8), c)
    def forward(self, x):
        w = torch.sigmoid(self.fc2(F.silu(self.fc1(x.mean(2)))))
        return x * w.unsqueeze(2)

class Block(nn.Module):
    def __init__(self, cin, cout, k, stride):
        super().__init__()
        self.c1 = nn.Conv1d(cin, cout, k, stride, k // 2, bias=False); self.b1 = nn.BatchNorm1d(cout)
        self.c2 = nn.Conv1d(cout, cout, k, 1, k // 2, bias=False);     self.b2 = nn.BatchNorm1d(cout)
        self.se = SE(cout)
        self.short = None
        if stride != 1 or cin != cout:
            self.short = nn.Sequential(nn.AvgPool1d(stride, stride, ceil_mode=True) if stride > 1 else nn.Identity(),
                                       nn.Conv1d(cin, cout, 1, bias=False), nn.BatchNorm1d(cout))
    def forward(self, x):
        h = F.silu(self.b1(self.c1(x)))
        h = self.se(self.b2(self.c2(h)))
        s = x if self.short is None else self.short(x)
        if s.shape[-1] != h.shape[-1]:
            s = s[..., :h.shape[-1]]
        return F.silu(h + s)

class AttnStats(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.att = nn.Sequential(nn.Linear(c, c // 2), nn.Tanh(), nn.Linear(c // 2, 1))
    def forward(self, h):                       # h : (B, T, C)
        w = torch.softmax(self.att(h), dim=1)
        mu = (w * h).sum(1)
        sd = torch.sqrt((w * (h - mu.unsqueeze(1)) ** 2).sum(1) + 1e-5)
        return torch.cat([mu, sd], dim=1)

class FilterBranch(nn.Module):
    """Filtres FIR appris (init aléatoire) sur I/Q normalisés + détecteur d'énergie, puis 3 étages stride 2."""
    def __init__(self, c_out, n_f=FB_F, k=FB_K, stride=FB_STRIDE):
        super().__init__()
        self.conv = nn.Conv1d(2, 2 * n_f, k, stride, (k - stride) // 2, bias=False)
        self.bn = nn.BatchNorm1d(n_f)
        self.b1, self.b2, self.b3 = Block(n_f, 96, 5, 2), Block(96, 128, 3, 2), Block(128, c_out, 3, 2)
    def forward(self, iq):
        a, b = self.conv(iq).chunk(2, dim=1)
        h = F.silu(self.bn(torch.sqrt(a * a + b * b + 1e-6)))
        return self.b3(self.b2(self.b1(h)))

class TrackANet(nn.Module):
    def __init__(self, n_in=N_CH, widths=(48, 64, 96, 128, 192, 256), blocks=(1, 1, 2, 2, 2, 2),
                 kernels=(7, 7, 5, 5, 3, 3), gru=128, drop=0.2, n_cls=7):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(n_in, 32, 31, 2, 15, bias=False), nn.BatchNorm1d(32), nn.SiLU())
        layers, c = [], 32
        for w, nb, k in zip(widths, blocks, kernels):
            for j in range(nb):
                layers.append(Block(c, w, k, 2 if j == 0 else 1)); c = w
        self.body = nn.Sequential(*layers)
        self.fb = FilterBranch(c) if USE_FB else None
        c_gru = 2 * c if USE_FB else c
        self.gru = nn.GRU(c_gru, gru, batch_first=True, bidirectional=True)
        self.ln = nn.LayerNorm(2 * gru)
        self.pool = AttnStats(2 * gru)
        self.head = nn.Sequential(nn.Dropout(drop), nn.Linear(4 * gru, 256), nn.SiLU(), nn.Dropout(drop), nn.Linear(256, n_cls))
    def forward(self, x):
        h = self.body(self.stem(x))                    # (B, C, T)
        if self.fb is not None:
            g = self.fb(x[:, :2])
            T = min(h.shape[-1], g.shape[-1])
            h = torch.cat([h[..., :T], g[..., :T]], dim=1)
        h = h.transpose(1, 2)                          # (B, T, C)
        h, _ = self.gru(h)
        return self.head(self.pool(self.ln(h)))

print("paramètres par modèle:", sum(p.numel() for p in TrackANet().parameters()))

paramètres par modèle: 3148012


## 5. Entraînement (curriculum de SNR, EMA, un modèle par graine)
`evaluate` accepte un modèle ou une liste de modèles (moyenne des probabilités). Pendant l'entraînement on évalue l'EMA sur un sous-ensemble fixe de la validation toutes les `EVAL_EVERY` époques.

In [5]:
@torch.no_grad()
def ema_update(ema, model, decay):
    for pe, pm in zip(ema.parameters(), model.parameters()):
        pe.mul_(decay).add_(pm.detach(), alpha=1 - decay)
    for be, bm in zip(ema.buffers(), model.buffers()):
        be.copy_(bm)

@torch.no_grad()
def evaluate(nets, X, y, npw, spw, snrs=SNRS, bs=512, seed=123):
    """Balanced accuracy par SNR (bruit à graine fixe). `nets` : un modèle ou une liste (ensemble)."""
    nets = list(nets) if isinstance(nets, (list, tuple)) else [nets]
    for n in nets: n.eval()
    gen = torch.Generator(device=dev); gen.manual_seed(seed)
    res = {}
    for snr in snrs:
        preds = []
        for i in range(0, len(X), bs):
            xb = add_noise(X[i:i+bs].float(), npw[i:i+bs], spw[i:i+bs], float(snr), gen)
            f = features(xb)
            with torch.autocast(device_type=dev.type, dtype=torch.bfloat16, enabled=dev.type == "cuda"):
                probs = sum(torch.softmax(n(f).float(), 1) for n in nets)
            preds.append(probs.argmax(1))
        preds = torch.cat(preds)
        rec = [(preds[y == c] == c).float().mean().item() for c in range(7) if (y == c).any()]
        res[snr] = float(np.mean(rec))
    return res

def sample_snr(n, epoch):
    """SNR par impulsion : curriculum (le SNR minimum descend de ~+6 à -30 dB), puis mélange centré sur les SNR bas."""
    lo = max(-30.0, 10.0 - 40.0 * min(1.0, epoch / CURR_EPOCHS))
    if epoch < CURR_EPOCHS:
        return torch.empty(n, device=dev).uniform_(lo, 30.0)
    u = torch.rand(n, device=dev)
    low = torch.empty(n, device=dev).uniform_(-30.0, -10.0)
    rest = torch.empty(n, device=dev).uniform_(-10.0, 30.0)
    return torch.where(u < LOW_FOCUS, low, rest)

# sous-ensemble de validation fixe pour le suivi rapide
g = torch.Generator().manual_seed(0)
sub = torch.randperm(len(Xva_t), generator=g)[:VAL_SUBSET].to(dev)
steps_per_epoch = len(Xtr_t) // BATCH

def train_one(seed, tag):
    torch.manual_seed(seed); np.random.seed(seed)
    model = TrackANet().to(dev)
    ema = copy.deepcopy(model).eval()
    for p in ema.parameters(): p.requires_grad_(False)
    decay, no_decay = [], []
    for n_, p_ in model.named_parameters():
        (no_decay if p_.ndim <= 1 else decay).append(p_)
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": WEIGHT_DECAY}, {"params": no_decay, "weight_decay": 0.0}], lr=LR)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * steps_per_epoch, pct_start=0.1)
    best, best_state, step = -1.0, None, 0
    t_start = time.time()
    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm = torch.randperm(len(Xtr_t), device=dev)
        tot, n_log, t1 = 0.0, 0, time.time()
        for s in range(steps_per_epoch):
            idx = perm[s*BATCH:(s+1)*BATCH]
            snr = sample_snr(len(idx), epoch)
            xb = add_noise(Xtr_t[idx].float(), ntr_t[idx], str_t[idx], snr)
            yb = ytr_t[idx]
            feats = features(xb, rotate=True)
            with torch.autocast(device_type=dev.type, dtype=torch.bfloat16, enabled=dev.type == "cuda"):
                logits = model(feats)
            loss = F.cross_entropy(logits.float() + LOGIT_ADJ * LOG_PRIOR, yb, label_smoothing=LABEL_SMOOTH)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            opt.step(); sched.step(); step += 1
            ema_update(ema, model, min(EMA_DECAY, (1 + step) / (10 + step)))
            if s % 20 == 0:
                tot += loss.item(); n_log += 1
        msg = f"[{tag}] epoch {epoch:02d}/{EPOCHS}  loss~{tot / max(1, n_log):.3f}  ({time.time()-t1:.0f}s)"
        if epoch % EVAL_EVERY == 0 or epoch == EPOCHS:
            res = evaluate(ema, Xva_t[sub], yva_t[sub], nva_t[sub], sva_t[sub])
            score = float(np.mean(list(res.values())))
            msg += f"  | val(sous-ens.) mean BA {score:.4f}  [-15:{res[-15]:.2f} -20:{res[-20]:.2f} -25:{res[-25]:.2f} -30:{res[-30]:.2f}]"
            if score > best:
                best, best_state = score, copy.deepcopy(ema.state_dict())
                torch.save(best_state, OUT_DIR / f"checkpoint_{tag}.pt")
        print(msg, flush=True)
    print(f"[{tag}] fini en {(time.time()-t_start)/60:.1f} min | meilleur score sous-ensemble: {best:.4f}")
    return best_state, best

members, member_files = [], []
for k, seed in enumerate(SEEDS):
    state, sc = train_one(seed, f"m{k}")
    net = TrackANet().to(dev); net.load_state_dict(state); net.eval()
    torch.save(state, OUT_DIR / f"model_{k}.pt")        # sauvegardé dès la fin de chaque modèle
    members.append(net); member_files.append(f"model_{k}.pt")

[m0] epoch 01/80  loss~0.786  (57s)
[m0] epoch 02/80  loss~0.281  (44s)
[m0] epoch 03/80  loss~0.265  (45s)
[m0] epoch 04/80  loss~0.268  (44s)  | val(sous-ens.) mean BA 0.7050  [-15:0.22 -20:0.15 -25:0.14 -30:0.14]
[m0] epoch 05/80  loss~0.265  (49s)
[m0] epoch 06/80  loss~0.275  (51s)
[m0] epoch 07/80  loss~0.318  (49s)
[m0] epoch 08/80  loss~0.310  (54s)  | val(sous-ens.) mean BA 0.7583  [-15:0.47 -20:0.26 -25:0.14 -30:0.14]
[m0] epoch 09/80  loss~0.364  (55s)
[m0] epoch 10/80  loss~0.420  (54s)
[m0] epoch 11/80  loss~0.497  (55s)
[m0] epoch 12/80  loss~0.705  (55s)  | val(sous-ens.) mean BA 0.7785  [-15:0.48 -20:0.32 -25:0.27 -30:0.17]
[m0] epoch 13/80  loss~0.681  (54s)
[m0] epoch 14/80  loss~0.667  (55s)
[m0] epoch 15/80  loss~0.663  (55s)
[m0] epoch 16/80  loss~0.665  (54s)  | val(sous-ens.) mean BA 0.8320  [-15:0.80 -20:0.50 -25:0.34 -30:0.20]
[m0] epoch 17/80  loss~0.652  (55s)
[m0] epoch 18/80  loss~0.659  (54s)
[m0] epoch 19/80  loss~0.638  (49s)
[m0] epoch 20/80  loss~0.639

## 6. Évaluation complète sur la validation officielle
13 200 impulsions × 13 SNR : chaque modèle seul, puis l'ensemble. C'est le chiffre à comparer à la v1 (0.8656) et à Track B (≈0.89).

In [6]:
def report(name, res):
    m = float(np.mean(list(res.values()))); t = float(np.mean([res[-20], res[-25], res[-30]]))
    print(f"{name:10s} mean BA {m:.4f} | tie-break (-20/-25/-30) {t:.4f} | " + " ".join(f"{s}:{res[s]:.2f}" for s in (-10, -15, -20, -25, -30)))
    return m, t

for k, net in enumerate(members):
    report(f"model_{k}", evaluate(net, Xva_t, yva_t, nva_t, sva_t, seed=2024))
full = evaluate(members, Xva_t, yva_t, nva_t, sva_t, seed=2024)
mean_ba, tiebreak = report("ENSEMBLE", full)
for s, v in full.items():
    print(f"  {s:+4d} dB : {v:.3f}")

model_0    mean BA 0.8881 | tie-break (-20/-25/-30) 0.5171 | -10:1.00 -15:0.99 -20:0.84 -25:0.48 -30:0.24
model_1    mean BA 0.8891 | tie-break (-20/-25/-30) 0.5213 | -10:1.00 -15:0.99 -20:0.85 -25:0.48 -30:0.24
model_2    mean BA 0.8885 | tie-break (-20/-25/-30) 0.5189 | -10:1.00 -15:0.99 -20:0.85 -25:0.47 -30:0.24
ENSEMBLE   mean BA 0.8914 | tie-break (-20/-25/-30) 0.5305 | -10:1.00 -15:1.00 -20:0.86 -25:0.49 -30:0.24
   +30 dB : 1.000
   +25 dB : 1.000
   +20 dB : 1.000
   +15 dB : 1.000
   +10 dB : 1.000
    +5 dB : 1.000
    +0 dB : 1.000
    -5 dB : 1.000
   -10 dB : 1.000
   -15 dB : 0.996
   -20 dB : 0.860
   -25 dB : 0.487
   -30 dB : 0.244


## 6b. Diagnostic : où se perdent les points ?
Matrices de confusion (lignes = vrai, colonnes = prédit, en %) à quelques SNR. **Envoie-moi cette sortie** pour cibler la v4.


In [7]:
@torch.no_grad()
def confusion(nets, X, y, npw, spw, snr, bs=512, seed=7):
    nets = list(nets) if isinstance(nets, (list, tuple)) else [nets]
    for n in nets: n.eval()
    gen = torch.Generator(device=dev); gen.manual_seed(seed)
    preds = []
    for i in range(0, len(X), bs):
        xb = add_noise(X[i:i+bs].float(), npw[i:i+bs], spw[i:i+bs], float(snr), gen)
        f = features(xb)
        with torch.autocast(device_type=dev.type, dtype=torch.bfloat16, enabled=dev.type == "cuda"):
            probs = sum(torch.softmax(n(f).float(), 1) for n in nets)
        preds.append(probs.argmax(1))
    preds = torch.cat(preds).cpu().numpy(); yy = y.cpu().numpy()
    cm = np.zeros((7, 7))
    for t, p in zip(yy, preds): cm[t, p] += 1
    return 100 * cm / cm.sum(1, keepdims=True)

for snr in (0, -10, -20, -30):
    cm = confusion(members, Xva_t, yva_t, nva_t, sva_t, snr)
    print(f"\n=== SNR {snr:+d} dB (lignes=vrai, colonnes=prédit) ===")
    print("        " + " ".join(f"{c[:6]:>6s}" for c in CLASSES))
    for i, c in enumerate(CLASSES):
        print(f"{c[:7]:7s} " + " ".join(f"{v:6.1f}" for v in cm[i]))



=== SNR +0 dB (lignes=vrai, colonnes=prédit) ===
        barker     cw   fmcw    fsk    hyp   nlfm   quad
barker   100.0    0.0    0.0    0.0    0.0    0.0    0.0
cw         0.0  100.0    0.0    0.0    0.0    0.0    0.0
fmcw       0.0    0.0  100.0    0.0    0.0    0.0    0.0
fsk        0.0    0.0    0.0  100.0    0.0    0.0    0.0
hyp        0.0    0.0    0.0    0.0  100.0    0.0    0.0
nlfm       0.0    0.0    0.0    0.0    0.0  100.0    0.0
quad       0.0    0.0    0.0    0.0    0.0    0.0  100.0

=== SNR -10 dB (lignes=vrai, colonnes=prédit) ===
        barker     cw   fmcw    fsk    hyp   nlfm   quad
barker   100.0    0.0    0.0    0.0    0.0    0.0    0.0
cw         0.0  100.0    0.0    0.0    0.0    0.0    0.0
fmcw       0.0    0.0  100.0    0.0    0.0    0.0    0.0
fsk        0.0    0.0    0.0  100.0    0.0    0.0    0.0
hyp        0.0    0.0    0.0    0.0  100.0    0.0    0.0
nlfm       0.0    0.0    0.0    0.0    0.0  100.0    0.0
quad       0.0    0.0    0.0    0.0    0.0  

## 7. Sauvegarde des livrables (poids, config, hash, résumé)

In [8]:
config = dict(track="A", features="RMS-normalised I,Q + lag conjugate products + 2nd-order lag products + learned FIR filterbank branch (energy detector)", lags=list(LAGS), haf_pairs=[list(p) for p in HAF_PAIRS] if USE_HAF else [], use_haf=USE_HAF, use_fb=USE_FB, fb=[FB_F, FB_K, FB_STRIDE], crop_samples=[S0, S1],
              widths=[48, 64, 96, 128, 192, 256], blocks=[1, 1, 2, 2, 2, 2], kernels=[7, 7, 5, 5, 3, 3], gru=128, n_in=N_CH,
              models=member_files, seeds=SEEDS, epochs=EPOCHS, curriculum_epochs=CURR_EPOCHS, low_focus=LOW_FOCUS,
              classes=CLASSES, snrs=SNRS, val_mean_ba=mean_ba, val_tiebreak=tiebreak, val_per_snr={str(k): v for k, v in full.items()})
json.dump(config, open(OUT_DIR / "config.json", "w"), indent=2)
(OUT_DIR / "method_summary.md").write_text(f"""# Track A method summary
Strictly time-domain processing (no frequency-domain or time-frequency transform of any kind). Input: complex I/Q of the 21-119 us window, divided by per-pulse RMS.
Channels: I, Q, real/imag of z[n]*conj(z[n-k]) for k in {list(LAGS)}, and second-order products p_k[n]*conj(p_k[n-m]) for (k,m) in {list(HAF_PAIRS) if USE_HAF else []}. Second branch: learned FIR filterbank ({FB_F} complex filters, length {FB_K}, random init, stride {FB_STRIDE}) followed by magnitude (all learned time-domain convolutions; no FFT/STFT/wavelet).
Model: two-branch learned 1D conv (lag-product branch + filterbank branch) + residual SE conv blocks + bidirectional GRU + attentive statistics pooling + MLP; ensemble of {len(member_files)} models (seeds {SEEDS}), softmax averaged.
Training: official 80/20 split (training pulses only, 52,800), on-the-fly Gaussian noise at random SNR (curriculum over {CURR_EPOCHS} epochs, then {int(LOW_FOCUS*100)}% of draws in [-30,-10] dB), random carrier phase,
logit-adjusted label-smoothed CE (balanced accuracy), AdamW + OneCycle, EMA weights, bf16 autocast, {EPOCHS} epochs.
Validation mean balanced accuracy over the 13 SNRs: {mean_ba:.4f}; tie-break (-20/-25/-30 dB): {tiebreak:.4f}.
AI assistance: the code was written with help from an AI assistant (Claude); the team reviewed and ran it.
""")
sha = {f: hashlib.sha256((OUT_DIR / f).read_bytes()).hexdigest() for f in member_files + ["config.json"]}
(OUT_DIR / "SHA256SUMS").write_text("".join(f"{h}  {f}\n" for f, h in sha.items()))
print("Livrables dans", OUT_DIR, "\n", json.dumps(sha, indent=1))

Livrables dans C:\Users\Think pro\Desktop\Radar_ML_Lite_DATATHON\Radar_ML_Lite_supervised_multiclass_classification\Radar_ML_lite\track_a_runs\final_v3 
 {
 "model_0.pt": "eaa2893170a210ecd0bb09111c1789bd97c5bbcd9177beaff44303dc0e5e3a97",
 "model_1.pt": "a318ed5358c431362f4e999ce01e7cfb92a1b06aaa7be91c82b82f88845f65ef",
 "model_2.pt": "7f540175c854b80d240747fb77ee850bca51bf913bbd72071f1f1300c663b2bf",
 "config.json": "a50cc7efbe63d2dd483707dbea59411f4514ba38a87269cb6edf6f65ccc800eb"
}


## 8. Prédiction sur le test aveugle
Ne se fait plus dans le notebook : utilise le script d'inférence qui gère l'ensemble et écrit l'en-tête exigé `sample_id,label` :

```
python src\track_a\predict_track_a.py --selftest
python src\track_a\predict_track_a.py
```

*(fin du notebook)*